In [1]:
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.utils import shuffle
from tqdm import tqdm
import numpy as np

In [2]:
try:
    data = pd.read_csv("Churn.csv")
except Exception:
    data = pd.read_csv("/datasets/Churn.csv")

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB


*Не хватает 10% данных в столбце Tenure*

In [4]:
print(data.head(20))

    RowNumber  CustomerId    Surname  CreditScore Geography  Gender  Age  \
0           1    15634602   Hargrave          619    France  Female   42   
1           2    15647311       Hill          608     Spain  Female   41   
2           3    15619304       Onio          502    France  Female   42   
3           4    15701354       Boni          699    France  Female   39   
4           5    15737888   Mitchell          850     Spain  Female   43   
5           6    15574012        Chu          645     Spain    Male   44   
6           7    15592531   Bartlett          822    France    Male   50   
7           8    15656148     Obinna          376   Germany  Female   29   
8           9    15792365         He          501    France    Male   44   
9          10    15592389         H?          684    France    Male   27   
10         11    15767821     Bearce          528    France    Male   31   
11         12    15737173    Andrews          497     Spain    Male   24   
12         1

*Не вижу взаимосвязи между кредитным рейтингом, колличеством используемых продуктов, балансом и столбцом Tenure, поэтому удалю пропуски*

In [5]:
data = data.dropna(subset=["Tenure"]).reset_index(drop=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9091 entries, 0 to 9090
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        9091 non-null   int64  
 1   CustomerId       9091 non-null   int64  
 2   Surname          9091 non-null   object 
 3   CreditScore      9091 non-null   int64  
 4   Geography        9091 non-null   object 
 5   Gender           9091 non-null   object 
 6   Age              9091 non-null   int64  
 7   Tenure           9091 non-null   float64
 8   Balance          9091 non-null   float64
 9   NumOfProducts    9091 non-null   int64  
 10  HasCrCard        9091 non-null   int64  
 11  IsActiveMember   9091 non-null   int64  
 12  EstimatedSalary  9091 non-null   float64
 13  Exited           9091 non-null   int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 994.5+ KB


In [6]:
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data = pd.get_dummies(data, drop_first=True)
data = data.sample(frac=1)

In [7]:
target = data['Exited']
features = data.drop('Exited', axis=1)

In [8]:
features_t, features_valid, target_t, target_valid = train_test_split(
    features, target, test_size=0.25, random_state=12345, stratify=target)

features_train, features_test, target_train, target_test = train_test_split(
    features_t, target_t, test_size=0.25, random_state=12345, stratify=target_t)


In [9]:
threshold = 0.32

In [10]:
best_model_forest = None
best_f1_score = 0
best_precision = 0
best_recall = 0
best_auc_roc = 0

In [11]:
for i in tqdm(range(1, 100)):
    for g in range(3, 20):
        model = RandomForestClassifier(random_state=12345, n_estimators=i, max_depth=g)
        model.fit(features_train, target_train)
        
        probabilities = model.predict_proba(features_valid)
        probabilities_one = probabilities[:, 1]
        prediction_valid = probabilities_one > threshold

        precision = precision_score(target_valid, prediction_valid)
        recall = recall_score(target_valid, prediction_valid)
        f1 = f1_score(target_valid, prediction_valid)
        roc_auc = roc_auc_score(target_valid, prediction_valid)

        if f1 > best_f1_score:
            best_f1_score = f1
            best_precision = precision
            best_recall = recall
            best_model_forest = model
            best_auc_roc = roc_auc

100%|██████████| 99/99 [07:56<00:00,  4.81s/it]


In [12]:
print(f"Best F1 = {'%.4f' % (best_f1_score)}\nAUC-ROC = {'%.4f' % (best_auc_roc)}\nPecision = {'%.4f' % (best_precision)}\nRecall = {'%.4f' % (best_recall)}")

Best F1 = 0.6279
AUC-ROC = 0.7689
Pecision = 0.6162
Recall = 0.6401


*Extra Trees Classifier*

In [92]:
%%time
model_extra_trees = ExtraTreesClassifier(random_state=12345, max_depth=2000, n_estimators=1000)
model_extra_trees.fit(features_train, target_train)

probabilities_extra_trees = model_extra_trees.predict_proba(features_valid)
probabilities_extra_trees_one_valid = probabilities_extra_trees[:, 1]

predicted_extra_trees_valid = probabilities_extra_trees_one_valid > 0.36
precision_extra_trees = precision_score(target_valid, predicted_extra_trees_valid)
recall_extra_trees = recall_score(target_valid, predicted_extra_trees_valid)
f1_extra_trees = f1_score(target_valid, predicted_extra_trees_valid)
roc_auc_extra_trees = roc_auc_score(target_valid, probabilities_extra_trees_one_valid)

print(f"Best F1 = {'%.4f' % (f1_extra_trees)}\nAUC-ROC = {'%.4f' % (roc_auc_extra_trees)}\nPecision = {'%.4f' % (precision_extra_trees)}\nRecall = {'%.4f' % (recall_extra_trees)}")


Best F1 = 0.5816
AUC-ROC = 0.8431
Pecision = 0.5835
Recall = 0.5797
CPU times: user 4.66 s, sys: 8.03 ms, total: 4.67 s
Wall time: 4.67 s


*Best model*

In [90]:
probabilities = best_model_forest.predict_proba(features_test)
probabilities_one = probabilities[:, 1]
prediction_test = probabilities_one > threshold

precision = precision_score(target_test, prediction_test)
recall = recall_score(target_test, prediction_test)
f1 = f1_score(target_test, prediction_test)
roc_auc = roc_auc_score(target_test, prediction_test)

print(f"Best F1 = {'%.4f' % (f1)}\nAUC-ROC = {'%.4f' % (roc_auc)}\nPecision = {'%.4f' % (precision)}\nRecall = {'%.4f' % (recall)}")


Best F1 = 0.5877
AUC-ROC = 0.7391
Pecision = 0.5982
Recall = 0.5776


*Weighted*

In [59]:
%%time
model_weigted = RandomForestClassifier(random_state=12345, n_estimators=100, max_depth=20, class_weight="balanced")
model_weigted.fit(features_train, target_train)
        
probabilities_weigted = model_weigted.predict_proba(features_valid)
probabilities_one_weigted = probabilities_weigted[:, 1]
prediction_weigted_valid = probabilities_one_weigted > 0.38
precision_weigted = precision_score(target_valid, prediction_weigted_valid)
recall_weigted = recall_score(target_valid, prediction_weigted_valid)
f1_weigted = f1_score(target_valid, prediction_weigted_valid)
roc_auc_weigted = roc_auc_score(target_valid, prediction_weigted_valid)

print(f"Best F1 = {'%.4f' % (f1_weigted)}\nAUC-ROC = {'%.4f' % (roc_auc_weigted)}\nPecision = {'%.4f' % (precision_weigted)}\nRecall = {'%.4f' % (recall_weigted)}")


Best F1 = 0.6041
AUC-ROC = 0.7454
Pecision = 0.6357
Recall = 0.5754
CPU times: user 639 ms, sys: 0 ns, total: 639 ms
Wall time: 641 ms


In [89]:
probabilities_weigted = model_weigted.predict_proba(features_test)
probabilities_one_weigted = probabilities_weigted[:, 1]
prediction_weigted_test = probabilities_one_weigted > 0.38
precision_weigted = precision_score(target_test, prediction_weigted_test)
recall_weigted = recall_score(target_test, prediction_weigted_test)
f1_weigted = f1_score(target_test, prediction_weigted_test)
roc_auc_weigted = roc_auc_score(target_test, prediction_weigted_test)

print(f"Best F1 = {'%.4f' % (f1_weigted)}\nAUC-ROC = {'%.4f' % (roc_auc_weigted)}\nPecision = {'%.4f' % (precision_weigted)}\nRecall = {'%.4f' % (recall_weigted)}")

Best F1 = 0.5852
AUC-ROC = 0.7276
Pecision = 0.6642
Recall = 0.5230


*Upsempling*

In [17]:
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345)
    
    return features_upsampled, target_upsampled

In [18]:
features_upsampled, target_upsampled = upsample(features_train, target_train, 4)

In [84]:
%%time
model_upsemled = RandomForestClassifier(random_state=12345, n_estimators=1000, max_depth=20)
model_upsemled.fit(features_upsampled, target_upsampled)
        
probabilities_upsemled = model_upsemled.predict_proba(features_valid)
probabilities_one_upsemled = probabilities_upsemled[:, 1]
prediction_upsemled_valid = probabilities_one_upsemled > 0.38
precision_upsemled = precision_score(target_valid, prediction_upsemled_valid)
recall_upsemled = recall_score(target_valid, prediction_upsemled_valid)
f1_upsemled = f1_score(target_valid, prediction_upsemled_valid)
roc_auc_upsemled = roc_auc_score(target_valid, prediction_upsemled_valid)

print(f"Best F1 = {'%.4f' % (f1_upsemled)}\nAUC-ROC = {'%.4f' % (roc_auc_upsemled)}\nPecision = {'%.4f' % (precision_upsemled)}\nRecall = {'%.4f' % (recall_upsemled)}")

Best F1 = 0.5906
AUC-ROC = 0.7549
Pecision = 0.5391
Recall = 0.6530
CPU times: user 9.25 s, sys: 56 ms, total: 9.31 s
Wall time: 9.31 s


In [86]:
probabilities_upsemled = model_upsemled.predict_proba(features_test)
probabilities_one_upsemled = probabilities_upsemled[:, 1]
prediction_upsemled_test = probabilities_one_upsemled > 0.38
precision_upsemled = precision_score(target_test, prediction_upsemled_test)
recall_upsemled = recall_score(target_test, prediction_upsemled_test)
f1_upsemled = f1_score(target_test, prediction_upsemled_test)
roc_auc_upsemled = roc_auc_score(target_test, prediction_upsemled_test)

print(f"Best F1 = {'%.4f' % (f1_upsemled)}\nAUC-ROC = {'%.4f' % (roc_auc_upsemled)}\nPecision = {'%.4f' % (precision_upsemled)}\nRecall = {'%.4f' % (recall_upsemled)}")

Best F1 = 0.5865
AUC-ROC = 0.7473
Pecision = 0.5536
Recall = 0.6236


**Weighting is better then upsempling**

**По какой-то причине все равно разные результаты получаю**

**По результатам - у меня по какой-то причине при одинаковых аргументах разные величины при каждой загрузке на проверку**

*Вывод:
1)Лучшей моделью оказался простой лес
2)По предобработке взвешенные классы показали лучше всего, однако пришлось менять гиперпараметры
3)По точности взвешенные классы превосходят метод upsemple, однако уступают на те же 10% по полноте, соответственно точность предсказаний и приеилемой полнотой лучше здесь
4)Было бы лучше, если бы получилось сделать минимальную полноту 0.6+ и точность 0.7+*